In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "ecom_poc"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_files"
BATCH_ID = "initial_load_batch_001"

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA ecom_poc")

def ingest_to_bronze(file_name: str, target_table: str):
    file_path = f"{VOLUME_PATH}/{file_name}"
    
    # 1. Read raw CSV strictly as STRING without automatic schema inference
    raw_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(file_path)
    )
    
    # Explicitly cast every column to STRING
    string_cols = [F.col(c).cast("string").alias(c) for c in raw_df.columns]
    
    # 2. Add lineage and audit metadata columns
    bronze_df = (
        raw_df.select(*string_cols)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.lit(file_path))
        .withColumn("_batch_id", F.lit(BATCH_ID))
        .withColumn(
            "_row_hash",
            F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c), F.lit("NULL")) for c in raw_df.columns]), 256)
        )
    )
    
    # 3. Idempotent write: replace only this batch_id so re-runs never duplicate records
    if spark.catalog.tableExists(target_table):
        (
            bronze_df.write
            .format("delta")
            .mode("overwrite")
            .option("replaceWhere", f"_batch_id = '{BATCH_ID}'")
            .saveAsTable(target_table)
        )
    else:
        bronze_df.write.format("delta").mode("overwrite").saveAsTable(target_table)
        
    cnt = spark.table(target_table).count()
    print(f"Loaded {target_table}: {cnt:,} rows (Batch: {BATCH_ID})")

ingest_to_bronze("orders.csv", "bronze_orders")
ingest_to_bronze("transactions.csv", "bronze_transactions")

Loaded bronze_orders: 1,000,000 rows (Batch: initial_load_batch_001)
Loaded bronze_transactions: 1,000,000 rows (Batch: initial_load_batch_001)


In [0]:
%sql
USE workspace.ecom_poc;

-- 1. Verify exact 1,000,000 count parity
SELECT 'bronze_orders' AS table_name, COUNT(*) AS row_count FROM bronze_orders
UNION ALL
SELECT 'bronze_transactions' AS table_name, COUNT(*) AS row_count FROM bronze_transactions;

-- 2. Verify all source columns are STRING
DESCRIBE TABLE bronze_orders;

-- 3. Inspect sample rows with audit metadata
SELECT * FROM bronze_orders LIMIT 5;

order_id,customer_id,order_date,category,quantity,unit_price,order_status,country,_ingested_at,_source_file,_batch_id,_row_hash
ORD-0868792,CUST-076802,2026-01-02,Fashion,1,9083.45,DELIVERED,Bangladesh,2026-11-18T09:26:38.746Z,/Volumes/workspace/ecom_poc/raw_files/orders.csv,initial_load_batch_001,dd3c1c851d92b25cc955f038b42444653f5b40fb6573608668998f0384942d80
ORD-0868793,CUST-126925,2026-06-03,Groceries,8,7991.81,CANCELLED,Bangladesh,2026-11-18T09:26:38.746Z,/Volumes/workspace/ecom_poc/raw_files/orders.csv,initial_load_batch_001,02a08d87f347e49cd35e70e54f13e8b6305c044c323002e1f3b4a9edb72c3c2e
ORD-0868794,CUST-000711,2026-07-02,Fashion,8,10368.3,DELIVERED,Bangladesh,2026-11-18T09:26:38.746Z,/Volumes/workspace/ecom_poc/raw_files/orders.csv,initial_load_batch_001,feb0e759c6c6a4c8cf4762924da4bc8aa616044f7606f48adf9ea29cc7704cba
ORD-0868795,CUST-128144,2026-07-27,Electronics,4,11337.71,DELIVERED,Bangladesh,2026-11-18T09:26:38.746Z,/Volumes/workspace/ecom_poc/raw_files/orders.csv,initial_load_batch_001,5c41c89ad318af9439b225b23f9ed172b2ba1181507f31363c5be8126049cc13
ORD-0868796,CUST-144378,2026-09-01,Home & Kitchen,9,2049.84,DELIVERED,Bangladesh,2026-11-18T09:26:38.746Z,/Volumes/workspace/ecom_poc/raw_files/orders.csv,initial_load_batch_001,6bb5dc20f7b463239b851a47dc8ef6e9836ff53f7bff34f0f49be84ce246c639
